# Gold → ML

| Parte | Entrada | Saída |
|---|---|---|
| Previsão de demanda | `ml_demanda_semanal` | `ml_metricas_demanda`, `ml_backtest_demanda`, `ml_previsao_demanda` |
| Sugestão de compra | `ml_previsao_demanda`, `fact_estoque_diario`, `fact_compra` | `ml_sugestao_compra` |
| Segmentação de itens | `ml_perfil_item` | `ml_segmento_item` |
| Consumo anômalo | `ml_demanda_semanal` | `ml_anomalia_consumo` |

Previsão por item e semana, somando os locais, com horizonte de 4 semanas. Treino até o corte e teste nas 13 semanas seguintes, comparado a dois baselines.

## 1. Parâmetros

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

dbutils.widgets.text("catalogo", "estoque_hospitalar", "Catálogo")
CATALOGO = dbutils.widgets.get("catalogo").strip()
spark.sql(f"USE CATALOG {CATALOGO}")

HORIZONTE = 4
SEMANAS_TESTE = 13
SEMENTE = 42


def gravar(pdf, tabela, comentario, colunas_data=()):
    df = spark.createDataFrame(pdf)
    for c in colunas_data:
        df = df.withColumn(c, F.to_date(c))
    (
        df.withColumn("_dt_processamento", F.current_timestamp())
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"gold.{tabela}")
    )
    spark.sql(f"COMMENT ON TABLE gold.{tabela} IS '{comentario}'")
    print(f"gold.{tabela:<26} {spark.table(f'gold.{tabela}').count():>10,}")

## 2. Previsão de demanda

Cada linha parte de uma semana de origem `o` e prevê a semana `o + h`, com `h` de 1 a 4. As features usam só dados até a origem, então um único modelo atende os quatro horizontes.

In [0]:
demanda = (
    spark.table("gold.ml_demanda_semanal")
    .filter("fl_semana_completa")
    .select("cd_material", "dt_inicio_semana", "qt_consumo", "nr_dias_ruptura", "vl_custo_medio",
            "nr_semana_ano", "nr_mes", "nr_feriados", "cd_grupo", "ds_classe_abc", "ds_classe_xyz")
    .toPandas()
)
demanda["dt_inicio_semana"] = pd.to_datetime(demanda["dt_inicio_semana"])
demanda = demanda.sort_values(["cd_material", "dt_inicio_semana"]).reset_index(drop=True)
demanda["ds_classe_abc"] = demanda["ds_classe_abc"].fillna("Sem consumo 12m")
demanda["ds_classe_xyz"] = demanda["ds_classe_xyz"].fillna("Z")

g = demanda.groupby("cd_material")["qt_consumo"]
origem = demanda[["cd_material", "dt_inicio_semana", "cd_grupo", "ds_classe_abc", "ds_classe_xyz"]].copy()
origem = origem.rename(columns={"dt_inicio_semana": "dt_origem"})
origem["qt_ultima"] = demanda["qt_consumo"]
for k in [1, 3, 7, 11]:
    origem[f"qt_lag_{k}"] = g.shift(k)
origem["qt_media_4"] = g.transform(lambda s: s.rolling(4).mean())
origem["qt_media_12"] = g.transform(lambda s: s.rolling(12).mean())
origem["qt_desvio_12"] = g.transform(lambda s: s.rolling(12).std())
origem["pc_semanas_com_consumo_12"] = demanda.assign(c=demanda["qt_consumo"] > 0).groupby("cd_material")["c"].transform(lambda s: s.rolling(12).mean())
origem["nr_dias_ruptura"] = demanda["nr_dias_ruptura"]
origem["vl_custo_medio"] = demanda.groupby("cd_material")["vl_custo_medio"].ffill()

calendario = demanda[["dt_inicio_semana", "nr_semana_ano", "nr_mes", "nr_feriados"]].drop_duplicates("dt_inicio_semana")
alvo = demanda[["cd_material", "dt_inicio_semana", "qt_consumo"]]

# Features da origem; alvo e calendário da semana prevista
linhas = []
for h in range(1, HORIZONTE + 1):
    parte = origem.copy()
    parte["nr_horizonte"] = h
    parte["dt_inicio_semana"] = parte["dt_origem"] + pd.Timedelta(weeks=h)
    linhas.append(parte)
dataset = (
    pd.concat(linhas, ignore_index=True)
    .merge(calendario, on="dt_inicio_semana", how="left")
    .merge(alvo, on=["cd_material", "dt_inicio_semana"], how="left")
    .dropna(subset=["qt_media_12"])
)

FEATURES = ["nr_horizonte", "qt_ultima", "qt_lag_1", "qt_lag_3", "qt_lag_7", "qt_lag_11", "qt_media_4",
            "qt_media_12", "qt_desvio_12", "pc_semanas_com_consumo_12", "nr_dias_ruptura", "vl_custo_medio",
            "nr_semana_ano", "nr_mes", "nr_feriados", "cd_grupo"]
ESCALA = ["qt_ultima", "qt_lag_1", "qt_lag_3", "qt_lag_7", "qt_lag_11", "qt_media_4", "qt_media_12",
          "qt_desvio_12", "vl_custo_medio"]


def matriz(df):
    # log1p aproxima itens de escalas muito diferentes num modelo único
    x = df[FEATURES].copy()
    x[ESCALA] = np.log1p(x[ESCALA].clip(lower=0))
    return x


def novo_modelo():
    return HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1.0, categorical_features=[FEATURES.index("cd_grupo")], random_state=SEMENTE,
    )


ultima_semana = demanda["dt_inicio_semana"].max()
corte = ultima_semana - pd.Timedelta(weeks=SEMANAS_TESTE)
print(f"Treino com alvo até {corte.date()}, teste de {(corte + pd.Timedelta(weeks=1)).date()} a {ultima_semana.date()}")
print(f"Linhas no dataset: {len(dataset):,}")

Treino com alvo até 2026-06-22, teste de 2026-06-29 a 2026-09-21
Linhas no dataset: 224,480


### 2.1 Backtest

In [0]:
treino = dataset[dataset["qt_consumo"].notna() & (dataset["dt_inicio_semana"] <= corte)]
teste = dataset[dataset["qt_consumo"].notna() & (dataset["dt_inicio_semana"] > corte)].copy()

# Alvo em log1p no treino; expm1 devolve a previsão para unidades
modelo = novo_modelo().fit(matriz(treino), np.log1p(treino["qt_consumo"].clip(lower=0)))

teste["qt_previsto_modelo"] = np.expm1(modelo.predict(matriz(teste))).clip(min=0)
teste["qt_previsto_ingenuo"] = teste["qt_ultima"]
teste["qt_previsto_media_4"] = teste["qt_media_4"]

MODELOS = {"modelo": "qt_previsto_modelo", "ingenuo": "qt_previsto_ingenuo", "media_4": "qt_previsto_media_4"}


def metricas(df, chaves):
    resultado = []
    for nome, coluna in MODELOS.items():
        m = (
            df.assign(erro_abs=(df[coluna] - df["qt_consumo"]).abs(), erro=df[coluna] - df["qt_consumo"])
            .groupby(chaves)
            .agg(qt_real=("qt_consumo", "sum"), erro_abs=("erro_abs", "sum"), erro=("erro", "sum"),
                 mae=("erro_abs", "mean"), nr_previsoes=("qt_consumo", "size"))
            .reset_index()
        )
        m["ds_modelo"] = nome
        # WAPE e viés só existem com consumo real no segmento
        m["pc_wape"] = np.where(m["qt_real"] > 0, m["erro_abs"] / m["qt_real"].where(m["qt_real"] > 0), np.nan)
        m["pc_vies"] = np.where(m["qt_real"] > 0, m["erro"] / m["qt_real"].where(m["qt_real"] > 0), np.nan)
        resultado.append(m.drop(columns=["erro_abs", "erro"]))
    return pd.concat(resultado, ignore_index=True)


geral = metricas(teste.assign(ds_segmento="Todos"), ["ds_segmento", "nr_horizonte"])
por_abc = metricas(teste.rename(columns={"ds_classe_abc": "ds_segmento"}), ["ds_segmento", "nr_horizonte"])
por_xyz = metricas(teste.assign(ds_segmento="XYZ " + teste["ds_classe_xyz"]), ["ds_segmento", "nr_horizonte"])
metricas_demanda = pd.concat([geral, por_abc, por_xyz], ignore_index=True)

display(
    metricas_demanda.pivot_table(index=["ds_segmento", "nr_horizonte"], columns="ds_modelo", values="pc_wape")
    .round(3).reset_index()
)

ds_segmento,nr_horizonte,ingenuo,media_4,modelo
A,1,0.225,0.174,0.175
A,2,0.225,0.17,0.175
A,3,0.232,0.174,0.174
A,4,0.225,0.17,0.173
B,1,0.339,0.269,0.267
B,2,0.335,0.258,0.264
B,3,0.362,0.267,0.263
B,4,0.317,0.261,0.261
C,1,0.282,0.237,0.221
C,2,0.288,0.239,0.225


In [0]:
gravar(metricas_demanda, "ml_metricas_demanda", "WAPE, MAE e viés do backtest por modelo, segmento e horizonte")

backtest = teste[["cd_material", "dt_origem", "dt_inicio_semana", "nr_horizonte", "ds_classe_abc", "ds_classe_xyz",
                  "qt_consumo", "qt_previsto_modelo", "qt_previsto_ingenuo", "qt_previsto_media_4"]]
gravar(backtest, "ml_backtest_demanda", "Previsto vs realizado nas semanas de teste",
       colunas_data=["dt_origem", "dt_inicio_semana"])

gold.ml_metricas_demanda                96
gold.ml_backtest_demanda            31,720


### 2.2 Previsão das próximas semanas

Modelo retreinado com todo o histórico. O intervalo de 80% usa o desvio do erro de cada item no backtest.

In [0]:
historico = dataset[dataset["qt_consumo"].notna()]
modelo_final = novo_modelo().fit(matriz(historico), np.log1p(historico["qt_consumo"].clip(lower=0)))

futuro = dataset[dataset["dt_origem"] == ultima_semana].copy()
futuro["qt_previsto"] = np.expm1(modelo_final.predict(matriz(futuro))).clip(min=0)

erro_item = (
    teste.assign(erro=teste["qt_previsto_modelo"] - teste["qt_consumo"])
    .groupby("cd_material")["erro"].std().rename("qt_desvio_erro").reset_index()
)
previsao = futuro.merge(erro_item, on="cd_material", how="left")
# Item sem erro no backtest usa o desvio das últimas 12 semanas
previsao["qt_desvio_erro"] = previsao["qt_desvio_erro"].fillna(previsao["qt_desvio_12"]).fillna(0)
previsao["qt_previsto_inferior"] = (previsao["qt_previsto"] - 1.28 * previsao["qt_desvio_erro"]).clip(lower=0)
previsao["qt_previsto_superior"] = previsao["qt_previsto"] + 1.28 * previsao["qt_desvio_erro"]

previsao = previsao[["cd_material", "dt_origem", "dt_inicio_semana", "nr_horizonte", "ds_classe_abc", "ds_classe_xyz",
                     "qt_previsto", "qt_previsto_inferior", "qt_previsto_superior", "qt_desvio_erro"]]
gravar(previsao, "ml_previsao_demanda", "Previsão de consumo das próximas 4 semanas por item, com intervalo de 80%",
       colunas_data=["dt_origem", "dt_inicio_semana"])

gold.ml_previsao_demanda             2,440


## 3. Sugestão de compra

| Parâmetro | Regra |
|---|---|
| Proteção | Prazo de entrega do fornecedor principal + 1 semana de revisão, até 4 semanas |
| Estoque de segurança | z do nível de serviço x desvio do erro de previsão x raiz das semanas de proteção |
| Nível de serviço | A 98%, B 95%, C 90% |
| Quantidade | Estoque alvo menos saldo atual, arredondada para a embalagem de compra |
| Sob demanda | Itens com saldo zero na maior parte dos últimos 90 dias não recebem reposição |

Sem pedidos em aberto na origem, o saldo em trânsito é considerado zero.

In [0]:
Z_SERVICO = {"A": 2.05, "B": 1.65, "C": 1.28}

dt_saldo = spark.table("gold.fact_estoque_diario").agg(F.max("dt_data")).first()[0]
saldo_atual = (
    spark.table("gold.fact_estoque_diario")
    .filter(F.col("dt_data") == dt_saldo)
    .groupBy("cd_material")
    .agg(F.sum(F.greatest("qt_saldo", F.lit(0.0))).alias("qt_saldo_atual"), F.sum("vl_saldo").alias("vl_saldo_atual"))
    .toPandas()
)
fornecedor_principal = (
    spark.table("gold.fact_compra")
    .groupBy("cd_material")
    .agg(F.max_by("cd_fornecedor", "dt_entrada").alias("cd_fornecedor"), F.max_by("vl_unitario_estoque", "dt_entrada").alias("vl_ultimo_custo"))
    .join(spark.table("gold.dim_fornecedor").select("cd_fornecedor", "qt_dias_entrega"), "cd_fornecedor", "left")
    .toPandas()
)
sob_demanda = (
    spark.table("gold.fact_estoque_diario")
    .filter(F.col("dt_data") > F.date_sub(F.lit(dt_saldo), 90))
    .groupBy("cd_material", "dt_data").agg(F.sum("qt_saldo").alias("qt_saldo"))
    .groupBy("cd_material").agg(F.avg((F.col("qt_saldo") > 0).cast("int")).alias("pc_dias_com_saldo_90d"))
    .toPandas()
)
material = spark.table("gold.dim_material").select("cd_material", "ds_material", "fl_ativo", "qt_conv_compra_estoque", "cd_unidade_compra").toPandas()

sugestao = (
    previsao.pivot_table(index="cd_material", columns="nr_horizonte", values="qt_previsto").add_prefix("qt_prev_s").reset_index()
    .merge(previsao.groupby("cd_material").agg(qt_desvio_erro=("qt_desvio_erro", "first"), ds_classe_abc=("ds_classe_abc", "first")).reset_index(), on="cd_material")
    .merge(saldo_atual, on="cd_material", how="left")
    .merge(fornecedor_principal, on="cd_material", how="left")
    .merge(material, on="cd_material", how="left")
    .merge(sob_demanda, on="cd_material", how="left")
)
sugestao = sugestao[sugestao["fl_ativo"].fillna(False)].copy()
sugestao["qt_saldo_atual"] = sugestao["qt_saldo_atual"].fillna(0)
sugestao["qt_dias_entrega"] = sugestao["qt_dias_entrega"].fillna(7)

sugestao["nr_semanas_protecao"] = np.minimum(np.ceil(sugestao["qt_dias_entrega"] / 7) + 1, HORIZONTE).astype(int)
colunas_prev = [f"qt_prev_s{h}" for h in range(1, HORIZONTE + 1)]
prev = sugestao[colunas_prev].to_numpy()
# Soma só as semanas dentro da proteção de cada item
mascara = np.arange(1, HORIZONTE + 1)[None, :] <= sugestao["nr_semanas_protecao"].to_numpy()[:, None]
sugestao["qt_demanda_protecao"] = (prev * mascara).sum(axis=1)

z = sugestao["ds_classe_abc"].map(Z_SERVICO).fillna(Z_SERVICO["C"])
sugestao["qt_estoque_seguranca"] = z * sugestao["qt_desvio_erro"] * np.sqrt(sugestao["nr_semanas_protecao"])
sugestao["qt_estoque_alvo"] = sugestao["qt_demanda_protecao"] + sugestao["qt_estoque_seguranca"]

embalagem = sugestao["qt_conv_compra_estoque"].fillna(1).clip(lower=1)
demanda_media = sugestao[colunas_prev].mean(axis=1)
# Embalagem que cobre mais de 8 semanas de demanda é trocada por compra unitária
embalagem = embalagem.where(embalagem <= 8 * demanda_media, 1)
sugestao["qt_embalagem_compra"] = embalagem
necessidade = (sugestao["qt_estoque_alvo"] - sugestao["qt_saldo_atual"]).clip(lower=0)
sugestao["ds_politica"] = np.where(sugestao["pc_dias_com_saldo_90d"].fillna(1) < 0.5, "Sob demanda", "Reposição")
sugestao["qt_embalagens"] = np.where(sugestao["ds_politica"] == "Reposição", np.ceil(necessidade / embalagem), 0)
sugestao["qt_sugerida_estoque"] = sugestao["qt_embalagens"] * embalagem
sugestao["vl_estimado"] = (sugestao["qt_sugerida_estoque"] * sugestao["vl_ultimo_custo"]).round(2)
sugestao["qt_semanas_cobertura_atual"] = np.where(
    sugestao["qt_prev_s1"] > 0, sugestao["qt_saldo_atual"] / sugestao["qt_prev_s1"], np.nan)
sugestao["fl_comprar"] = sugestao["qt_embalagens"] > 0
sugestao["dt_referencia"] = pd.Timestamp(dt_saldo)

sugestao = sugestao[[
    "dt_referencia", "cd_material", "ds_material", "ds_classe_abc", "ds_politica", "cd_fornecedor", "qt_dias_entrega",
    "nr_semanas_protecao", "qt_saldo_atual", "qt_semanas_cobertura_atual", *colunas_prev, "qt_demanda_protecao",
    "qt_estoque_seguranca", "qt_estoque_alvo", "cd_unidade_compra", "qt_conv_compra_estoque", "qt_embalagem_compra", "qt_embalagens",
    "qt_sugerida_estoque", "vl_ultimo_custo", "vl_estimado", "fl_comprar",
]]
gravar(sugestao, "ml_sugestao_compra", "Quantidade sugerida de compra por item a partir da previsão de demanda",
       colunas_data=["dt_referencia"])

display(
    sugestao.groupby(["ds_politica", "ds_classe_abc"])
    .agg(itens=("cd_material", "size"), itens_comprar=("fl_comprar", "sum"), valor_estimado=("vl_estimado", "sum"))
    .round(0).reset_index()
)

gold.ml_sugestao_compra                594


ds_politica,ds_classe_abc,itens,itens_comprar,valor_estimado
Reposição,A,86,36,188762.0
Reposição,B,162,30,24913.0
Reposição,C,335,8,2226.0
Sob demanda,A,11,0,0.0


## 4. Segmentação de itens

K-Means com volume, valor e custo em log antes da padronização. Número de grupos escolhido pelo maior silhouette entre 3 e 7.

In [0]:
perfil = spark.table("gold.ml_perfil_item").toPandas()
perfil = perfil[perfil["qt_consumo_total"] > 0].copy()

atributos = pd.DataFrame({
    "log_valor_consumo": np.log1p(perfil["vl_consumo_total"].clip(lower=0)),
    "log_custo_unitario": np.log1p(perfil["vl_custo_medio"].fillna(perfil["vl_custo_medio"].median())),
    "pc_semanas_com_consumo": perfil["pc_semanas_com_consumo"],
    "nr_cv_semanal": perfil["nr_cv_semanal"].fillna(perfil["nr_cv_semanal"].max()).clip(upper=5),
    "log_cobertura": np.log1p(perfil["qt_dias_cobertura_medio"].fillna(perfil["qt_dias_cobertura_medio"].median()).clip(lower=0)),
})
x = StandardScaler().fit_transform(atributos)

silhuetas = {}
for k in range(3, 8):
    rotulos = KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit_predict(x)
    silhuetas[k] = silhouette_score(x, rotulos)
k_escolhido = max(silhuetas, key=silhuetas.get)
print({k: round(v, 3) for k, v in silhuetas.items()}, "escolhido:", k_escolhido)

perfil["nr_cluster"] = KMeans(n_clusters=k_escolhido, n_init=20, random_state=SEMENTE).fit_predict(x)

{3: 0.411, 4: 0.379, 5: 0.35, 6: 0.286, 7: 0.284} escolhido: 3


In [0]:
resumo = perfil.groupby("nr_cluster").agg(
    itens=("cd_material", "size"),
    vl_consumo_mediano=("vl_consumo_total", "median"),
    vl_custo_mediano=("vl_custo_medio", "median"),
    pc_semanas_com_consumo=("pc_semanas_com_consumo", "median"),
    nr_cv_semanal=("nr_cv_semanal", "median"),
    qt_dias_cobertura=("qt_dias_cobertura_medio", "median"),
).reset_index()


def nivel(valor_cluster, valores_itens):
    # Compara a mediana do cluster com os tercis de todos os itens, e não com os outros clusters
    t1, t2 = np.nanquantile(valores_itens, [1 / 3, 2 / 3])
    return np.select([valor_cluster <= t1, valor_cluster <= t2], ["baixo", "médio"], "alto")


resumo["ds_segmento"] = (
    "Valor " + nivel(resumo["vl_consumo_mediano"], perfil["vl_consumo_total"])
    + ", custo unitário " + nivel(resumo["vl_custo_mediano"], perfil["vl_custo_medio"])
    + ", consumo " + np.where(resumo["pc_semanas_com_consumo"] >= 0.7, "regular", "intermitente")
)
# Clusters com o mesmo perfil recebem o número do cluster para não se confundirem
duplicado = resumo["ds_segmento"].duplicated(keep=False)
resumo.loc[duplicado, "ds_segmento"] += " (" + resumo.loc[duplicado, "nr_cluster"].astype(str) + ")"
display(resumo.round(2))

segmento = perfil.merge(resumo[["nr_cluster", "ds_segmento"]], on="nr_cluster")
segmento = segmento.merge(
    spark.table("gold.mart_curva_abc_xyz").select("cd_material", "ds_classe_abc", "ds_classe_xyz").toPandas(),
    on="cd_material", how="left",
)
segmento = segmento[["cd_material", "ds_material", "ds_grupo", "nr_cluster", "ds_segmento", "ds_classe_abc", "ds_classe_xyz",
                     "vl_consumo_total", "vl_custo_medio", "pc_semanas_com_consumo", "nr_cv_semanal",
                     "qt_dias_cobertura_medio", "pc_perda"]]
gravar(segmento, "ml_segmento_item", "Segmentação de itens por K-Means sobre consumo, custo, regularidade e cobertura")

display(pd.crosstab(segmento["ds_segmento"], segmento["ds_classe_abc"].fillna("-")).reset_index())

nr_cluster,itens,vl_consumo_mediano,vl_custo_mediano,pc_semanas_com_consumo,nr_cv_semanal,qt_dias_cobertura,ds_segmento
0,134,1224.3,6.84,0.4,1.6,101.08,"Valor baixo, custo unitário médio, consumo intermitente"
1,384,12576.46,4.26,1.0,0.37,68.44,"Valor médio, custo unitário médio, consumo regular"
2,92,24908.39,475.5,0.3,1.76,47.91,"Valor alto, custo unitário alto, consumo intermitente"


gold.ml_segmento_item                  610


ds_segmento,-,A,B,C
"Valor alto, custo unitário alto, consumo intermitente",4,28,40,20
"Valor baixo, custo unitário médio, consumo intermitente",2,0,5,127
"Valor médio, custo unitário médio, consumo regular",0,69,118,197


## 5. Consumo anômalo

Itens com consumo em pelo menos metade das semanas. Anomalia: desvio acima de 5 MAD em relação à mediana das 12 semanas anteriores e diferença de pelo menos 50% sobre o esperado. Quedas em semanas com ruptura são marcadas como possível falta de estoque.

In [0]:
LIMITE_MAD = 5
LIMITE_RELATIVO = 0.5

regulares = perfil.loc[perfil["pc_semanas_com_consumo"] >= 0.5, "cd_material"]
serie = demanda[demanda["cd_material"].isin(regulares)].copy()
# shift(1) para a referência usar só as semanas anteriores
anterior = serie.groupby("cd_material")["qt_consumo"].shift(1)
grupo_anterior = anterior.groupby(serie["cd_material"])

serie["qt_esperado"] = grupo_anterior.transform(lambda s: s.rolling(12).median())
serie["qt_mad"] = grupo_anterior.transform(lambda s: s.rolling(12).apply(lambda w: np.median(np.abs(w - np.median(w))), raw=True))
# 1.4826 x MAD equivale ao desvio padrão em distribuição normal; piso evita divisão por MAD zero
escala = np.maximum(1.4826 * serie["qt_mad"], np.maximum(1.0, 0.1 * serie["qt_esperado"]))
serie["nr_score"] = (serie["qt_consumo"] - serie["qt_esperado"]) / escala

relativo = (serie["qt_consumo"] - serie["qt_esperado"]).abs() / serie["qt_esperado"].clip(lower=1)
anomalias = serie[(serie["nr_score"].abs() > LIMITE_MAD) & (relativo >= LIMITE_RELATIVO)].copy()
anomalias["ds_tipo"] = np.where(anomalias["nr_score"] > 0, "Alta", "Baixa")
anomalias["fl_possivel_ruptura"] = (anomalias["ds_tipo"] == "Baixa") & (anomalias["nr_dias_ruptura"] > 0)
anomalias["qt_desvio"] = anomalias["qt_consumo"] - anomalias["qt_esperado"]
anomalias["vl_desvio_estimado"] = (anomalias["qt_desvio"] * anomalias["vl_custo_medio"]).round(2)

anomalias = anomalias[["cd_material", "dt_inicio_semana", "ds_tipo", "qt_consumo", "qt_esperado", "qt_desvio",
                       "nr_score", "vl_desvio_estimado", "nr_dias_ruptura", "fl_possivel_ruptura", "ds_classe_abc"]]
gravar(anomalias, "ml_anomalia_consumo", "Semanas com consumo fora do padrão do item, para auditoria",
       colunas_data=["dt_inicio_semana"])

display(anomalias.groupby(["ds_tipo", "fl_possivel_ruptura"]).size().reset_index(name="semanas"))

gold.ml_anomalia_consumo               433


ds_tipo,fl_possivel_ruptura,semanas
Alta,false,394
Baixa,false,12
Baixa,true,27
